In [1]:
# ============================================================
# Cell 1: Environment Initialisation & Grad-CAM Setup
# ============================================================
import os
import sys
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torchvision
from torchvision import transforms
from PIL import Image

# Install pytorch-grad-cam if not installed
try:
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    print("Installing pytorch-grad-cam...")
    !pip install grad-cam
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Cell 1: XAI Environment Initialized")
print(f"PyTorch Version   : {torch.__version__}")
print(f"Compute Device    : {device}")
if torch.cuda.is_available():
    print(f"GPU Model         : {torch.cuda.get_device_name(0)}")
print("=" * 70)

Cell 1: XAI Environment Initialized
PyTorch Version   : 2.13.0+cu126
Compute Device    : cuda
GPU Model         : NVIDIA GeForce RTX 3050


In [2]:
# ============================================================
# Cell 2: Paths & Project Configuration
# ============================================================
PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")

# Data Directories
DATA_DIR = PROJECT_ROOT / "data set" / "raw" / "chest_xray"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data set" / "processed"

TEST_DIR = DATA_DIR / "test"

# Weights and Outputs
CHECKPOINT_DIR = PROJECT_ROOT / "models"
MODEL_WEIGHT_PATH = CHECKPOINT_DIR / "PneumoXNet_best.pth"
XAI_OUTPUT_DIR = PROJECT_ROOT / "results" / "xai_heatmaps"
XAI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 260
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("=" * 70)
print("Cell 2: Paths & Directories Verified")
print(f"Model Checkpoint Path : {MODEL_WEIGHT_PATH} (Exists: {MODEL_WEIGHT_PATH.exists()})")
print(f"XAI Output Directory : {XAI_OUTPUT_DIR}")
print("=" * 70)

Cell 2: Paths & Directories Verified
Model Checkpoint Path : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/PneumoXNet_best.pth (Exists: True)
XAI Output Directory : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/results/xai_heatmaps


In [3]:
# ============================================================
# Cell 3: Load PneumoXNet Architecture & Hydrate Checkpoint
# ============================================================
import timm
import torch.nn as nn

class PneumoXNet(nn.Module):
    def __init__(self, num_classes=2, pretrained=False, dropout_rate=0.4):
        super(PneumoXNet, self).__init__()
        self.backbone = timm.create_model('tf_efficientnet_b2.ns_jft_in1k', pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=dropout_rate / 2),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

# Initialize and load model
model = PneumoXNet(num_classes=2, pretrained=False).to(device)
checkpoint = torch.load(MODEL_WEIGHT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("=" * 70)
print("Cell 3: PneumoXNet Model Loaded Successfully")
print(f"Checkpoint Epoch      : {checkpoint['epoch']}")
print(f"Validation ROC-AUC    : {checkpoint['val_auc']:.4f}")
print("=" * 70)

Cell 3: PneumoXNet Model Loaded Successfully
Checkpoint Epoch      : 20
Validation ROC-AUC    : 1.0000


In [4]:
# ============================================================
# Cell 4: Target Convolutional Layer Hook & Image Transforms
# ============================================================
# Identify the last convolutional layer of EfficientNet-B2 backbone for CAM extraction
target_layers = [model.backbone.conv_head]

# Image Pipeline (Normalization & RGB Conversion)
def preprocess_image(img_path, target_size=(260, 260)):
    pil_img = Image.open(img_path).convert('RGB')
    resized_img = pil_img.resize(target_size)
    
    # Numpy normalized array [0, 1] for visualization overlay
    rgb_img = np.float32(resized_img) / 255.0
    
    # Tensor Transformation for Model Prediction
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    input_tensor = transform(resized_img).unsqueeze(0).to(device)
    
    return pil_img, rgb_img, input_tensor

print("Cell 4: Target Conv Layer Hooked -> model.backbone.conv_head")

Cell 4: Target Conv Layer Hooked -> model.backbone.conv_head


In [5]:
# ============================================================
# Cell 5: Grad-CAM & Grad-CAM++ Generator Functions
# ============================================================
def generate_xai_heatmaps(model, target_layers, input_tensor, rgb_img, target_category=None):
    """
    Generates Grad-CAM and Grad-CAM++ heatmaps overlaid on the original image.
    """
    targets = [ClassifierOutputTarget(target_category)] if target_category is not None else None
    
    # 1. Grad-CAM Execution
    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_gradcam = cam(input_tensor=input_tensor, targets=targets)[0, :]
        cam_image = show_cam_on_image(rgb_img, grayscale_gradcam, use_rgb=True)

    # 2. Grad-CAM++ Execution (Better Localization for Multiple Lesions)
    with GradCAMPlusPlus(model=model, target_layers=target_layers) as cam_plus:
        grayscale_gradcam_pp = cam_plus(input_tensor=input_tensor, targets=targets)[0, :]
        cam_plus_image = show_cam_on_image(rgb_img, grayscale_gradcam_pp, use_rgb=True)

    return grayscale_gradcam, cam_image, grayscale_gradcam_pp, cam_plus_image

print("Cell 5: Grad-CAM and Grad-CAM++ Engine Functions Compiled")

Cell 5: Grad-CAM and Grad-CAM++ Engine Functions Compiled
